# Taller — Sesión 2: La ficha técnica de mi dataset

**Curso:** Aprendizaje Automático · Ciclo 3
**Sesión:** 2 — 22 de agosto de 2026
**Docente:** Carlos Giovanny Hidalgo Suárez · cghidalgos@usbcali.edu.co
**Universidad de San Buenaventura Cali — Facultad de Ingeniería**

---

## Cómo trabajar

Taller **individual**, sobre **tu propio dataset**. Son 4 ejercicios cortos.
Cada uno trae un ejemplo ya resuelto antes de que te toque a ti.

La sesión tiene **dos partes**, y se hacen en momentos distintos de la clase:

| Parte | Cuándo | Qué es | Puntos |
|---|---|---|---|
| **A** | Después de ver las categorías de ML | Tu **propuesta de proyecto** | **40** |
| **B1** | Después del bloque de preprocesamiento | Radiografía de tu dataset | 15 |
| **B2** | " | Diagnóstico de faltantes | 15 |
| **B3** | " | Inventario de variables | 15 |
| **B4** | " | Ficha técnica final | 15 |
| | | **Total** | **100** |

> ### ⚠️ La entrega de hoy es la Parte A + la Parte B
> No es tarea para después. Sales de clase con todo terminado y revisado.

## ¿No trajiste dataset?

La celda de abajo crea uno de respaldo (registros académicos ficticios pero
realistas: tiene nulos, duplicados, categóricas y desbalance).
**Úsalo solo si es indispensable** — el objetivo del curso es que trabajes con
datos tuyos.


In [20]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

print('Todo listo.')

Todo listo.


In [21]:
def cargar_dataset_respaldo(n=1500, semilla=42):
    """Crea un dataset académico de respaldo, con los problemas típicos de
    un dataset real: valores faltantes, duplicados, categóricas y desbalance."""
    rng = np.random.default_rng(semilla)

    df = pd.DataFrame({
        'id_estudiante':     np.arange(1, n + 1),
        'edad':              rng.integers(17, 35, n),
        'ciudad':            rng.choice(['Cali', 'Bogotá', 'Medellín', 'Palmira'],
                                        n, p=[.55, .2, .15, .10]),
        'estrato':           rng.choice([1, 2, 3, 4, 5], n, p=[.15, .3, .3, .15, .10]),
        'programa':          rng.choice(['Sistemas', 'Industrial', 'Civil', 'Electrónica'], n),
        'promedio_anterior': np.round(rng.normal(3.6, 0.5, n).clip(0, 5), 2),
        'creditos_inscritos': rng.integers(9, 21, n),
        'horas_trabajo':     rng.integers(0, 45, n).astype(float),
        'asistencia_pct':    np.round(rng.normal(82, 12, n).clip(0, 100), 1),
        'tiene_beca':        rng.choice(['Si', 'No'], n, p=[.25, .75]),
    })

    # La desercion depende de verdad de las variables (no es ruido puro)
    riesgo = (
        (df['promedio_anterior'] < 3.2) * 0.35 +
        (df['asistencia_pct'] < 70) * 0.30 +
        (df['horas_trabajo'] > 30) * 0.15 +
        (df['tiene_beca'] == 'No') * 0.05 + 0.04
    )
    df['deserta'] = (rng.random(n) < riesgo).astype(int)

    # Valores faltantes realistas
    df.loc[rng.choice(n, int(0.04 * n), replace=False), 'promedio_anterior'] = np.nan
    df.loc[rng.choice(n, int(0.31 * n), replace=False), 'horas_trabajo'] = np.nan
    df.loc[rng.choice(n, int(0.08 * n), replace=False), 'asistencia_pct'] = np.nan

    # Columna que solo existe si ya deserto  -> FUGA DE DATOS a proposito
    motivos = pd.Series(rng.choice(['Económico', 'Académico', 'Personal'], n),
                        index=df.index)
    df['motivo_retiro'] = motivos.where(df['deserta'] == 1)

    # Unas cuantas filas duplicadas
    df = pd.concat([df, df.sample(12, random_state=semilla)], ignore_index=True)
    return df.sample(frac=1, random_state=semilla).reset_index(drop=True)


# --- CARGA TU DATASET AQUI ---
# Opcion A (lo ideal): el tuyo
# datos = pd.read_csv('mi_archivo.csv')

# Opcion B (respaldo): descomenta la linea siguiente
datos = cargar_dataset_respaldo()

print('Dataset cargado:', datos.shape)
datos.head()

Dataset cargado: (1512, 12)


,id_estudiante,edad,ciudad,estrato,programa,promedio_anterior,creditos_inscritos,horas_trabajo,asistencia_pct,tiene_beca,deserta,motivo_retiro
0,908,20,Medellín,1,Civil,4.20,12,25.0,85.3,No,0,NaN
1,618,21,Cali,3,Industrial,4.44,16,11.0,100.0,No,0,NaN
2,1387,27,Cali,4,Civil,3.01,16,23.0,75.3,Si,0,NaN
3,942,24,Palmira,1,Sistemas,4.05,9,40.0,84.1,Si,0,NaN
4,304,33,Cali,5,Electrónica,3.60,16,3.0,75.4,No,0,NaN


---
---

# PARTE A — Tu propuesta de proyecto   *(40 puntos)*

> Esto se hace **después** del bloque de categorías del aprendizaje y **antes**
> del taller técnico. Ya sabes de qué tipo es tu problema; ahora escríbelo.

Un proyecto de datos no fracasa por el algoritmo: fracasa porque nadie escribió
con claridad qué se quería lograr. La propuesta es ese documento.

Son **cinco partes** y cabe en una hoja:

| # | Parte | En una línea |
|---|---|---|
| 1 | **Título** | Qué haces + sobre qué + con qué técnica |
| 2 | **Introducción** | Contexto · problema · tu propuesta (3 párrafos) |
| 3 | **Objetivo general** | Verbo + qué + cómo + para qué. **Uno solo** |
| 4 | **Objetivos específicos** | De 3 a 5, alineados con las entregas del curso |
| 5 | **Fuente de datos** | Fuente, enlace, licencia, tamaño, período, cómo se obtuvieron |

---

## A.1 · El título

**Fórmula:** `QUÉ HACES` + `SOBRE QUÉ` + `CON QUÉ TÉCNICA`

| ✗ No sirve | Por qué |
|---|---|
| «Machine Learning aplicado» | ¿Aplicado a qué? |
| «Proyecto de aprendizaje automático» | No dice absolutamente nada |
| «Análisis de datos con Python» | Python es la herramienta, no el tema |

> ✓ **Sí sirve:** «Sistema de alerta temprana de deserción estudiantil en pregrado
> mediante clasificación supervisada e integración con un modelo de lenguaje»

---

## A.2 · La introducción — Contexto general, contexto especifico, problematica, solucion propuesta, beneficios de la solución

1. **Contexto.** De qué se trata el ámbito y por qué le importa a alguien.
   Aquí van las cifras que tengas, **con su fuente**.
2. **El problema.** Qué está fallando hoy. Normalmente: la información existe
   pero nadie la usa a tiempo, o el problema se detecta cuando ya es tarde.
3. **Tu propuesta.** Qué vas a construir y con qué datos. La última frase
   debería anunciar tu objetivo general.

**No pongas aquí:** definiciones de qué es la IA, ni la historia del Machine
Learning. Eso ya lo sabemos.

---

## A.3 · El objetivo general

> **VERBO en infinitivo + QUÉ vas a lograr + CÓMO lo vas a hacer + PARA QUÉ sirve**

Uno solo. Si necesitas dos, es que tienes dos proyectos.

---

## A.4 · Los objetivos específicos — y aquí está el truco

Entre 3 y 5. Cada uno empieza con un **verbo en infinitivo** y se puede
verificar: o lo hiciste o no lo hiciste.

**El truco:** escríbelos de manera que coincidan con las entregas del curso.
Si lo haces bien, el cronograma trabaja para ti y llegas a la Sesión 9 con el
proyecto armado sin esfuerzo extra.

---

## A.5 · ¿De dónde saldrán los datos?

No basta decir «de Kaggle». Hay que poder responder seis cosas: **fuente,
enlace, licencia, tamaño, período** y **cómo se obtuvieron**.

- Si son datos de tu trabajo: necesitas **permiso** y **anonimización**.
- Si son de una entidad pública: cita el portal o la resolución.
- Si no puedes citar de dónde salieron, **no se aceptan**.

---

## Ejemplo completo

Ejecuta la celda siguiente para ver una propuesta terminada, con el nivel de
detalle que se espera.

In [22]:
# EJEMPLO COMPLETO — propuesta del proyecto de deserción
# (es el mismo caso del dataset de respaldo)

propuesta_ejemplo = {

 'titulo': (
    'Sistema de alerta temprana de deserción estudiantil en pregrado mediante '
    'clasificación supervisada e integración con un modelo de lenguaje'),

 'introduccion_1_contexto': (
    'La deserción en programas de pregrado representa una pérdida para el '
    'estudiante, que interrumpe su proyecto de vida, y para la institución, que '
    'pierde la inversión formativa ya realizada. Las universidades registran '
    'desde el primer semestre variables académicas y socioeconómicas de cada '
    'estudiante: notas, asistencia, créditos inscritos y situación laboral.'),

 'introduccion_2_problema': (
    'El problema no es la falta de información, sino que esa información se '
    'revisa tarde. Los comités de bienestar suelen detectar el riesgo cuando el '
    'estudiante ya dejó de asistir, es decir, cuando la intervención ya no es '
    'posible. Hoy la revisión es manual y depende de que un docente reporte el caso.'),

 'introduccion_3_propuesta': (
    'Este proyecto propone construir un modelo de clasificación que estime el '
    'riesgo de deserción de cada estudiante a partir de sus registros académicos '
    'del periodo en curso, y exponerlo como un servicio que además genere, '
    'mediante un modelo de lenguaje, una explicación en lenguaje natural de por '
    'qué se emitió cada alerta.'),

 'objetivo_general': (
    'Desarrollar un sistema de alerta temprana que identifique estudiantes en '
    'riesgo de deserción, integrando un modelo de clasificación supervisada con '
    'un modelo de lenguaje que explique cada alerta, para que bienestar '
    'universitario intervenga a tiempo.'),

 'objetivos_especificos': [
    ('Caracterizar y preparar el conjunto de datos académicos, resolviendo '
     'valores faltantes y codificación de variables categóricas.',        'S2 - S3'),
    ('Entrenar y evaluar modelos de clasificación, seleccionando el mejor '
     'según el recall sobre la clase en riesgo.',                          'S3 - S4'),
    ('Identificar perfiles de riesgo mediante técnicas de agrupación.',    'S5'),
    ('Exportar el modelo entrenado e implementarlo como un servicio de '
     'inferencia documentado.',                                            'S5 - S7'),
    ('Integrar el servicio con un modelo de lenguaje que genere una '
     'explicación de cada alerta.',                                        'S8'),
 ],

 'datos_fuente':      'Registros académicos institucionales (dataset de respaldo del curso)',
 'datos_enlace':      'generado en este notebook',
 'datos_licencia':    'uso académico',
 'datos_tamano':      '1512 filas x 12 columnas',
 'datos_periodo':     'un periodo académico',
 'datos_como':        'registro administrativo de matrícula y notas',
}


def mostrar_propuesta(p):
    """Imprime una propuesta con formato legible."""
    print('=' * 78)
    print('TÍTULO')
    print('=' * 78)
    print(' ', p['titulo'], '\n')

    print('=' * 78)
    print('INTRODUCCIÓN')
    print('=' * 78)
    for k in ['introduccion_1_contexto', 'introduccion_2_problema',
              'introduccion_3_propuesta']:
        print(' ', p[k], '\n')

    print('=' * 78)
    print('OBJETIVO GENERAL')
    print('=' * 78)
    print(' ', p['objetivo_general'], '\n')

    print('=' * 78)
    print('OBJETIVOS ESPECÍFICOS')
    print('=' * 78)
    for i, (o, ses) in enumerate(p['objetivos_especificos'], 1):
        print(f'  {i}. {o}   [{ses}]')
    print()

    print('=' * 78)
    print('FUENTE DE LOS DATOS')
    print('=' * 78)
    for etiqueta, clave in [('Fuente', 'datos_fuente'), ('Enlace', 'datos_enlace'),
                            ('Licencia', 'datos_licencia'), ('Tamaño', 'datos_tamano'),
                            ('Período', 'datos_periodo'), ('Cómo se obtuvieron', 'datos_como')]:
        print(f'  {etiqueta:20s}: {p[clave]}')


mostrar_propuesta(propuesta_ejemplo)

TÍTULO
  Sistema de alerta temprana de deserción estudiantil en pregrado mediante clasificación supervisada e integración con un modelo de lenguaje 

INTRODUCCIÓN
  La deserción en programas de pregrado representa una pérdida para el estudiante, que interrumpe su proyecto de vida, y para la institución, que pierde la inversión formativa ya realizada. Las universidades registran desde el primer semestre variables académicas y socioeconómicas de cada estudiante: notas, asistencia, créditos inscritos y situación laboral. 

  El problema no es la falta de información, sino que esa información se revisa tarde. Los comités de bienestar suelen detectar el riesgo cuando el estudiante ya dejó de asistir, es decir, cuando la intervención ya no es posible. Hoy la revisión es manual y depende de que un docente reporte el caso. 

  Este proyecto propone construir un modelo de clasificación que estime el riesgo de deserción de cada estudiante a partir de sus registros académicos del periodo en curso

> ### Fíjate en los objetivos específicos del ejemplo
> Los cinco están etiquetados con una sesión del curso (S2, S3, S5, S7, S8).
> No es casualidad: **son las entregas del syllabus**. Escribir así los objetivos
> convierte el cronograma del curso en el cronograma de tu proyecto.

---

## Ahora escribe la tuya

Copia la estructura y llénala con tu proyecto. Sé concreto.
Cuando termines, **levanta la mano** para revisarla antes de seguir con la Parte B.

In [23]:
# TODO — TU PROPUESTA (40 puntos)

mi_propuesta = {

 # 1. TÍTULO — qué haces + sobre qué + con qué técnica
 'titulo': 'Sistema de alerta temprana del riesgo reproductivo en ganado bovino mediante clasificación supervisada',

 # 2. INTRODUCCIÓN — tres párrafos
 'introduccion_1_contexto':  'La reproducción es muy importante en una ganadería, porque de esto depende que las vacas puedan tener crías de manera constante y que la finca pueda aprovechar mejor los recursos que utiliza en la alimentación, el manejo y el cuidado de los animales. En este proyecto se tomará como referencia que una vaca debería tener una nueva cría aproximadamente cada 12 a 14 meses. Para revisar este comportamiento se pueden utilizar los registros que ya tiene la Ganadería Olinda, como las fechas de parto, la edad de los animales, los antecedentes de maternidad y el tiempo que ha pasado desde el último parto',   # de qué se trata y por qué importa (con cifras si tienes)
 'introduccion_2_problema':  'El problema se presenta cuando algunas vacas pasan demasiado tiempo sin tener una nueva cría. Estos animales siguen generando gastos de alimentación, cuidado y mantenimiento, pero no están aportando la producción esperada para la ganadería. En la base de datos de la Ganadería Olinda ya se pueden observar vacas que llevan 14, 17, 18 e incluso 20 meses sin parir. Aunque estos registros permiten identificar cuáles animales ya tienen un problema reproductivo, muchas veces se detecta cuando la vaca ya superó el tiempo esperado. Por esta razón, es necesario analizar la información histórica para buscar una forma de identificar con anticipación cuáles vacas tienen mayor riesgo de superar los 14 meses entre partos.',   # qué está fallando hoy
 'introduccion_3_propuesta': 'Este proyecto propone desarrollar un modelo de clasificación supervisada utilizando los registros históricos de la Ganadería Olinda, con el objetivo de identificar las vacas que presentan mayor riesgo de tener un intervalo reproductivo superior a 14 meses. Para esto se pueden utilizar variables como la edad del animal, la fecha del último parto, los meses que lleva sin parir, sus antecedentes de maternidad y otras variables que puedan incorporarse posteriormente a la base de datos. El modelo permitirá clasificar las vacas según su nivel de riesgo y servirá como base para crear un sistema de alerta temprana. De esta manera, se podrán priorizar los animales que necesitan un seguimiento más cercano y tomar decisiones a tiempo para mejorar el manejo reproductivo y la productividad de la ganadería.',   # qué vas a construir y con qué datos

 # 3. OBJETIVO GENERAL — verbo + qué + cómo + para qué.  UNO SOLO.
 'objetivo_general': 'Desarrollar un sistema de alerta temprana que identifique vacas con riesgo de superar un intervalo de 14 meses entre partos, mediante un modelo de clasificación supervisada basado en información histórica reproductiva, para apoyar la toma de decisiones y mejorar la eficiencia reproductiva del hato.',

 # 4. OBJETIVOS ESPECÍFICOS — de 3 a 5, cada uno con la sesión donde lo cumples
 'objetivos_especificos': [
    ('Caracterizar y preparar el conjunto de datos reproductivos del ganado, identificando variables relevantes, valores faltantes, inconsistencias y variables categóricas. ', '[S4]?'),
    ('Construir variables e indicadores reproductivos a partir de los registros disponibles, tales como días desde el último parto, número de servicios e intervalos entre parto', '[S4]?'),
    ('Entrenar y evaluar modelos de clasificación supervisada que permitan estimar el riesgo de que una vaca supere los 14 meses entre partos', '[S6]?'),
    ('Implementar un sistema de alerta temprana que clasifique a las vacas según su nivel de riesgo y genere reportes para la toma de decisiones en el manejo reproductivo.', '[S8]?')
 ],

 # 5. ¿DE DÓNDE SALDRÁN LOS DATOS?
 'datos_fuente':   'Registros históricos reproductivos de ganado bovino de la Ganadería Olinda',
 'datos_enlace':   'No aplica inicialmente; información obtenida de registros propios de la explotación ganadera.',
 'datos_licencia': 'Uso académico autorizado por el propietario de los datos',
 'datos_tamano':   'Por determinar una vez se consoliden los registros históricos disponibles y los datos que se incorporen a traves de la alucinaciones',
 'datos_periodo':  ' Por determinar de acuerdo con los años disponibles en los registros reproductivos',
 'datos_como':     'Registros generados durante el manejo reproductivo de los animales, incluyendo información sobre identificación de cada vaca, fechas de parto,  diagnósticos de preñez y demás antecedentes reproductivos disponibles.',
}

mostrar_propuesta(mi_propuesta)

TÍTULO
  Sistema de alerta temprana del riesgo reproductivo en ganado bovino mediante clasificación supervisada 

INTRODUCCIÓN
  La reproducción es muy importante en una ganadería, porque de esto depende que las vacas puedan tener crías de manera constante y que la finca pueda aprovechar mejor los recursos que utiliza en la alimentación, el manejo y el cuidado de los animales. En este proyecto se tomará como referencia que una vaca debería tener una nueva cría aproximadamente cada 12 a 14 meses. Para revisar este comportamiento se pueden utilizar los registros que ya tiene la Ganadería Olinda, como las fechas de parto, la edad de los animales, los antecedentes de maternidad y el tiempo que ha pasado desde el último parto 

  El problema se presenta cuando algunas vacas pasan demasiado tiempo sin tener una nueva cría. Estos animales siguen generando gastos de alimentación, cuidado y mantenimiento, pero no están aportando la producción esperada para la ganadería. En la base de datos de l

In [24]:
# Autochequeo antes de levantar la mano.
# Ejecuta esta celda: te dice qué le falta a tu propuesta.

def revisar_propuesta(p):
    problemas = []
    if len(p.get('titulo', '')) < 40:
        problemas.append('El título es muy corto o está vacío: debe decir qué, sobre qué y con qué técnica.')
    for k, nombre in [('introduccion_1_contexto', 'contexto'),
                      ('introduccion_2_problema', 'problema'),
                      ('introduccion_3_propuesta', 'propuesta')]:
        if len(p.get(k, '')) < 80:
            problemas.append(f'El párrafo de {nombre} está vacío o es muy corto.')
    og = p.get('objetivo_general', '')
    if len(og) < 60:
        problemas.append('El objetivo general está vacío o es muy corto.')
    elif not og.strip().lower().startswith(('desarrollar', 'construir', 'implementar',
                                            'diseñar', 'crear', 'generar', 'establecer',
                                            'determinar', 'evaluar', 'predecir')):
        problemas.append('El objetivo general debería empezar con un verbo en infinitivo.')
    oes = [o for o, _ in p.get('objetivos_especificos', []) if o.strip()]
    if len(oes) < 3:
        problemas.append(f'Tienes {len(oes)} objetivos específicos; deben ser entre 3 y 5.')
    if len(oes) > 5:
        problemas.append(f'Tienes {len(oes)} objetivos específicos; son demasiados (máximo 5).')
    for campo in ['datos_fuente', 'datos_enlace', 'datos_licencia',
                  'datos_tamano', 'datos_periodo', 'datos_como']:
        if not p.get(campo, '').strip():
            problemas.append(f'Falta "{campo}" en la fuente de datos.')

    if problemas:
        print('Todavía te falta:\n')
        for i, x in enumerate(problemas, 1):
            print(f'  {i}. {x}')
    else:
        print('La propuesta está completa. Levanta la mano para que la revisemos.')
    return problemas


_ = revisar_propuesta(mi_propuesta)

La propuesta está completa. Levanta la mano para que la revisemos.


---
---

# PARTE B — La ficha técnica de tu dataset

> Se hace **después** del bloque de preprocesamiento.

La propuesta de la Parte A dice **qué quieres hacer**.
La ficha técnica dice **si tus datos te lo permiten**.

Cuatro ejercicios con pandas, 15 puntos cada uno.

### Mi dataset: Ganadería Olinda

Esta es la versión consolidada de mi dataset (`OLINDA_CLEAN.csv`): **2500 vacas**,
una fila por animal, con su historial reproductivo, físico y de salud. Es un
conjunto mucho más completo que el primer extracto que saqué a mano del Excel
de la finca (18 vacas) — aquí ya hay suficientes filas para entrenar un modelo
de verdad en la Sesión 3.

El objetivo binario es `candidata_descarte` (1 = la vaca es candidata a
descarte por bajo desempeño reproductivo). También viene `clasificacion_reproductiva`
(Excelente/Buena/Regular/Mala), una versión en cuatro categorías de la misma idea.

In [25]:
# --- CARGA DE MI DATASET: Ganadería Olinda ---
datos = pd.read_csv('OLINDA_CLEAN.csv', sep=';')

print('Dataset cargado:', datos.shape)
datos.head()

Dataset cargado: (2500, 15)


,vaca_id,edad_meses,color,cachona,peso_kg,condicion_corporal,estado_salud,edad_primer_parto_meses,num_partos,intervalo_partos_meses,meses_desde_ultimo_parto,perdida_cria,produccion_leche_lt_dia,candidata_descarte,clasificacion_reproductiva
0,S00001,134.5,amarillo,1,473.9,3.6,sana,35.8,7,14.8,3.4,0,7.9,0,Buena
1,S00002,144.0,negro,0,466.0,2.9,sana,37.9,8,15.9,1.6,1,6.3,1,Mala
2,S00003,76.7,amarillo,0,466.6,2.9,sana,38.8,3,11.2,2.2,0,8.1,0,Buena
3,S00004,187.7,pardo,0,495.1,3.6,sana,31.3,12,16.3,1.2,1,6.9,1,Regular
4,S00005,107.2,rojo,0,496.6,3.4,sana,27.5,6,14.7,5.9,0,7.8,0,Regular


---

## Ejercicio B1 — Radiografía de tu dataset   *(15 puntos)*

Antes de modelar nada hay que mirar los datos. Siempre las mismas seis líneas.

| Línea | Qué te responde |
|---|---|
| `.shape` | ¿Cuántas filas y columnas tengo? |
| `.head()` | ¿Cómo se ven los datos de verdad? |
| `.info()` | ¿Qué tipo tiene cada columna y cuántos nulos hay? |
| `.describe()` | ¿Los rangos tienen sentido? |
| `.duplicated().sum()` | ¿Hay filas repetidas? |
| `.value_counts()` | ¿Cómo está repartida la columna objetivo? |

### Ejemplo resuelto

Ejecuta la celda de abajo: es la radiografía del dataset de respaldo.

In [26]:
# EJEMPLO YA RESUELTO — solo ejecuta y observa
ejemplo = cargar_dataset_respaldo()

print('1. Forma      :', ejemplo.shape)
print('2. Duplicados :', ejemplo.duplicated().sum())
print()
print('3. Tipos y nulos:')
ejemplo.info()

1. Forma      : (1512, 12)
2. Duplicados : 12

3. Tipos y nulos:
<class 'pandas.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_estudiante       1512 non-null   int64  
 1   edad                1512 non-null   int64  
 2   ciudad              1512 non-null   str    
 3   estrato             1512 non-null   int64  
 4   programa            1512 non-null   str    
 5   promedio_anterior   1452 non-null   float64
 6   creditos_inscritos  1512 non-null   int64  
 7   horas_trabajo       1044 non-null   float64
 8   asistencia_pct      1390 non-null   float64
 9   tiene_beca          1512 non-null   str    
 10  deserta             1512 non-null   int64  
 11  motivo_retiro       367 non-null    str    
dtypes: float64(3), int64(5), str(4)
memory usage: 170.0 KB


**Cómo se lee esa salida:**

- `1512 filas, 12 columnas` — hay datos suficientes.
- `12 duplicados` — ojo: hay que decidir si son un error o son legítimos.
- En `.info()`, la columna `Non-Null Count` revela los faltantes:
  `horas_trabajo` tiene bastantes menos que las demás.
- `motivo_retiro` es de tipo `object` y está casi vacía. Ya volveremos sobre ella.

### La pregunta que vale más puntos

> ## ¿Qué representa **una fila** de tu dataset?

Se llama **unidad de observación**. En el dataset de respaldo, una fila es
**un estudiante**. Pero podría ser una matrícula, una transacción o una lectura
de sensor — y eso lo cambia todo:

Si hubiera **varias filas por estudiante**, no podrías partir train/test al azar,
porque el mismo estudiante quedaría en ambos lados y el modelo lo reconocería.

### Ahora tú

Haz la radiografía de **tu** dataset y responde.

In [27]:
# TODO 1.1 — la radiografía de TU dataset
print('Forma      :', datos.shape)
print('Duplicados :', datos.duplicated().sum())

# TODO 1.2 — tipos y nulos
print()
print('Tipos y nulos:')
datos.info()

# TODO 1.3 — ¿los rangos tienen sentido? ¿hay edades de 200 años?
print()
print('Rangos:')
print(datos.describe(include='all'))

# TODO 1.4 — reparto de tu columna objetivo (si tienes)
print()
print('Reparto de candidata_descarte:')
print(datos['candidata_descarte'].value_counts())

Forma      : (2500, 15)
Duplicados : 0

Tipos y nulos:
<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   vaca_id                     2500 non-null   str    
 1   edad_meses                  2500 non-null   float64
 2   color                       2500 non-null   str    
 3   cachona                     2500 non-null   int64  
 4   peso_kg                     2500 non-null   float64
 5   condicion_corporal          2500 non-null   float64
 6   estado_salud                2500 non-null   str    
 7   edad_primer_parto_meses     2500 non-null   float64
 8   num_partos                  2500 non-null   int64  
 9   intervalo_partos_meses      2500 non-null   float64
 10  meses_desde_ultimo_parto    2500 non-null   float64
 11  perdida_cria                2500 non-null   int64  
 12  produccion_leche_lt_dia     2500 non-null   fl

In [28]:
# TODO 1.5 — responde en texto (esto vale la mitad de los puntos del ejercicio)

unidad_de_observacion = 'una vaca de la Ganadería Olinda (una fila = un animal, identificado por vaca_id)'
hay_varias_filas_por_sujeto = False   # cada vaca_id aparece una sola vez
rangos_sospechosos = ('ninguno grave: edad_meses va de un becerro joven a una vaca vieja sin '
                      'valores absurdos (no hay edades de 0 ni de cientos de meses), '
                      'condicion_corporal se mueve en la escala esperada de 1 a 5, y '
                      'produccion_leche_lt_dia no tiene negativos')

print('Una fila es:', unidad_de_observacion)
print('¿Varias filas por sujeto?:', hay_varias_filas_por_sujeto)
print('Rangos sospechosos:', rangos_sospechosos)

Una fila es: una vaca de la Ganadería Olinda (una fila = un animal, identificado por vaca_id)
¿Varias filas por sujeto?: False
Rangos sospechosos: ninguno grave: edad_meses va de un becerro joven a una vaca vieja sin valores absurdos (no hay edades de 0 ni de cientos de meses), condicion_corporal se mueve en la escala esperada de 1 a 5, y produccion_leche_lt_dia no tiene negativos


---

## Ejercicio B2 — Diagnóstico de faltantes   *(15 puntos)*

Contar los nulos es lo fácil. Lo que se evalúa es **la decisión** que tomas con
cada columna y **cómo la justificas**.

Recuerda las estrategias vistas en clase:

| Estrategia | Cuándo es razonable |
|---|---|
| Eliminar la fila | Faltan pocas y al azar (< 5 %) |
| Eliminar la columna | Falta más del 40–50 % |
| Rellenar con media/mediana | Numérica, sin valores extremos |
| Rellenar con la moda | Categórica con una categoría dominante |
| Marcar como `"Desconocido"` | El hueco tiene significado propio |

### Ejemplo resuelto

In [29]:
# EJEMPLO YA RESUELTO — conteo de faltantes del dataset de respaldo
faltantes = pd.DataFrame({
    'nulos':     ejemplo.isna().sum(),
    'porcentaje': (ejemplo.isna().mean() * 100).round(1),
})
faltantes[faltantes['nulos'] > 0].sort_values('nulos', ascending=False)

,nulos,porcentaje
motivo_retiro,1145,75.7
horas_trabajo,468,31.0
asistencia_pct,122,8.1
promedio_anterior,60,4.0


**Las tres decisiones sobre este dataset, y por qué:**

| Columna | Nulos | Decisión | Razón |
|---|---|---|---|
| `promedio_anterior` | ~4 % | Rellenar con la mediana | Son pocos y no se ve un patrón |
| `horas_trabajo` | ~31 % | Marcar como `"No informa"` | Son demasiados para rellenar, y el hueco significa algo: no contestaron |
| `motivo_retiro` | ~82 % | **Eliminar la columna** | No solo por los nulos: **solo se llena si el estudiante YA desertó** |

> ### La tercera es la importante
> `motivo_retiro` es un caso de **fuga de datos**. Si la usara para predecir la
> deserción, mi modelo tendría casi 100 % de acierto... porque le estoy diciendo
> la respuesta. En producción esa columna está vacía para todos los estudiantes
> activos, que son justo los que quiero predecir.
>
> Esa columna no se elimina por tener muchos nulos. Se elimina **porque usarla
> sería hacer trampa**.

### Ahora tú

In [30]:
# TODO 2.1 — cuenta los faltantes de TU dataset
mis_faltantes = pd.DataFrame({
    'nulos':      datos.isna().sum(),
    'porcentaje': (datos.isna().mean() * 100).round(1),
})
mis_faltantes[mis_faltantes['nulos'] > 0]

,nulos,porcentaje


In [31]:
# TODO 2.2 — una decisión justificada por cada columna con nulos.
#            Añade o quita entradas según tu dataset.

mis_decisiones = {
    'ninguna_columna_tiene_nulos': {
        'porcentaje_nulos': 0.0,
        'decision':         'no aplica',
        'razon':            'Este dataset consolidado (OLINDA_CLEAN.csv) ya viene sin faltantes en '
                            'ninguna de las 15 columnas para las 2500 vacas; no hay que imputar ni '
                            'eliminar nada por esta razón.',
    },
}

# TODO 2.3 — ¿alguna columna de tu dataset sería FUGA DE DATOS?
#            (una que solo se conoce DESPUÉS de que pasó lo que quieres predecir)
posible_fuga = 'intervalo_partos_meses y clasificacion_reproductiva'
por_que      = ('candidata_descarte está construida principalmente a partir de '
                'intervalo_partos_meses (el promedio histórico de meses entre partos de esa vaca): '
                'cuando ese intervalo pasa de ~14-15 meses, casi siempre queda marcada como '
                'candidata a descarte. clasificacion_reproductiva es esa misma idea repartida en '
                'cuatro categorías (Excelente/Buena/Regular/Mala). Ninguna de las dos es "futuro" '
                'en sentido estricto, pero entrenar con ellas sería casi copiar la etiqueta en vez '
                'de aprender el patrón; las dejo fuera de X. meses_desde_ultimo_parto sí la '
                'conservo, porque es el dato que sí se conoce HOY sobre una vaca activa, antes de '
                'saber si va a superar el intervalo.')

pd.DataFrame(mis_decisiones).T

,porcentaje_nulos,decision,razon
ninguna_columna_tiene_nulos,0.0,no aplica,Este dataset consolidado (OLINDA_CLEAN.csv) ya...


---

## Ejercicio B3 — Inventario de variables   *(15 puntos)*

Clasifica cada columna. De esto depende cómo la vas a preprocesar en la Sesión 3.

| Tipo | Ejemplo | Qué se le hace |
|---|---|---|
| Numérica continua | promedio, edad, ingresos | Escalar (si el modelo lo necesita) |
| Categórica **sin** orden | ciudad, programa | **One-hot** |
| Categórica **con** orden | estrato, nivel educativo | **Ordinal** (0, 1, 2…) |
| Fecha | fecha_ingreso | Extraer año, mes, antigüedad |
| Identificador | cédula, id | **Se elimina**: no aporta y puede identificar |

> **La regla para decidir one-hot vs ordinal:** ¿tiene sentido decir que una
> categoría es *mayor* que otra? Estrato 4 > estrato 2, sí. ¿Medellín > Cali? No.

### Ejemplo resuelto

In [32]:
# EJEMPLO YA RESUELTO — inventario del dataset de respaldo
inventario = {
    'id_estudiante':      'identificador  -> eliminar',
    'edad':               'numérica continua',
    'ciudad':             'categórica SIN orden  -> one-hot',
    'estrato':            'categórica CON orden  -> ordinal',
    'programa':           'categórica SIN orden  -> one-hot',
    'promedio_anterior':  'numérica continua',
    'creditos_inscritos': 'numérica discreta',
    'horas_trabajo':      'numérica continua',
    'asistencia_pct':     'numérica continua',
    'tiene_beca':         'categórica binaria  -> 0/1',
    'deserta':            'VARIABLE OBJETIVO (categórica)',
    'motivo_retiro':      'eliminar: fuga de datos',
}
for col, tipo in inventario.items():
    print(f'{col:20s} {tipo}')

print('\n--- Balance de la variable objetivo ---')
print(ejemplo['deserta'].value_counts())
print()
print((ejemplo['deserta'].value_counts(normalize=True) * 100).round(1))

id_estudiante        identificador  -> eliminar
edad                 numérica continua
ciudad               categórica SIN orden  -> one-hot
estrato              categórica CON orden  -> ordinal
programa             categórica SIN orden  -> one-hot
promedio_anterior    numérica continua
creditos_inscritos   numérica discreta
horas_trabajo        numérica continua
asistencia_pct       numérica continua
tiene_beca           categórica binaria  -> 0/1
deserta              VARIABLE OBJETIVO (categórica)
motivo_retiro        eliminar: fuga de datos

--- Balance de la variable objetivo ---
deserta
0    1145
1     367
Name: count, dtype: int64

deserta
0    75.7
1    24.3
Name: proportion, dtype: float64


**Lee el balance con cuidado.** La clase minoritaria ronda el 24 %.

Eso significa que un modelo que diga *«nadie deserta»* acertaría el ~76 % de las
veces. Es exactamente el mito del 98 % de la Sesión 1, pero ahora **en tus datos**.

Por eso en la Sesión 3 no vamos a usar exactitud: usaremos recall, precisión y
matriz de confusión.

### Ahora tú

In [33]:
# TODO 3.1 — clasifica TODAS las columnas de tu dataset
mi_inventario = {
    'vaca_id':                     'identificador -> eliminar',
    'edad_meses':                  'numérica continua',
    'color':                       'categórica SIN orden -> one-hot',
    'cachona':                     'categórica binaria -> ya está en 0/1',
    'peso_kg':                     'numérica continua',
    'condicion_corporal':          'numérica continua (escala 1-5, pero se trata como numérica)',
    'estado_salud':                'categórica SIN orden (sana/nuche/enferma) -> one-hot',
    'edad_primer_parto_meses':     'numérica continua',
    'num_partos':                  'numérica discreta',
    'intervalo_partos_meses':      'numérica continua -> FUGA DE DATOS (define candidata_descarte)',
    'meses_desde_ultimo_parto':    'numérica continua',
    'perdida_cria':                'categórica binaria -> ya está en 0/1',
    'produccion_leche_lt_dia':     'numérica continua',
    'candidata_descarte':          'VARIABLE OBJETIVO (categórica binaria)',
    'clasificacion_reproductiva':  'categórica CON orden (Excelente>Buena>Regular>Mala) -> FUGA DE DATOS',
}

for col, tipo in mi_inventario.items():
    print(f'{col:28s} {tipo}')

# Ayuda: esto te lista los tipos que pandas detectó
# datos.dtypes

vaca_id                      identificador -> eliminar
edad_meses                   numérica continua
color                        categórica SIN orden -> one-hot
cachona                      categórica binaria -> ya está en 0/1
peso_kg                      numérica continua
condicion_corporal           numérica continua (escala 1-5, pero se trata como numérica)
estado_salud                 categórica SIN orden (sana/nuche/enferma) -> one-hot
edad_primer_parto_meses      numérica continua
num_partos                   numérica discreta
intervalo_partos_meses       numérica continua -> FUGA DE DATOS (define candidata_descarte)
meses_desde_ultimo_parto     numérica continua
perdida_cria                 categórica binaria -> ya está en 0/1
produccion_leche_lt_dia      numérica continua
candidata_descarte           VARIABLE OBJETIVO (categórica binaria)
clasificacion_reproductiva   categórica CON orden (Excelente>Buena>Regular>Mala) -> FUGA DE DATOS


In [34]:
# TODO 3.2 — el balance de tu variable objetivo
#            (si tu proyecto es de agrupación, escribe 'no aplica' y explica por qué)

print(datos['candidata_descarte'].value_counts())
print()
print((datos['candidata_descarte'].value_counts(normalize=True) * 100).round(1))

hay_desbalance = True
clase_minoritaria_pct = 32.5
que_implica = ('Con solo decir "ninguna es candidata a descarte" ya se acierta el 67.5% de las '
               'veces, así que la exactitud no sirve para juzgar el modelo. Toca mirar recall y '
               'precisión de la clase candidata_descarte = 1.')

candidata_descarte
0    1687
1     813
Name: count, dtype: int64

candidata_descarte
0    67.5
1    32.5
Name: proportion, dtype: float64


---

## Ejercicio B4 — Ficha técnica final   *(15 puntos)*

> ## ⚠️ Junto con la Parte A, esta es la ENTREGA de la Sesión 2
> No te vas de clase sin completarla y sin que yo la haya revisado.

Cierra tu propuesta con lo que descubriste al mirar los datos de verdad:
qué representa una fila, qué vas a hacer con los faltantes y las categóricas,
y qué columnas hay que dejar fuera.

### Ejemplo resuelto

In [35]:
# EJEMPLO YA RESUELTO — la ficha del dataset de respaldo

ficha_ejemplo = {
    'tema':               'Deserción estudiantil en pregrado',
    'dataset':            'Registros académicos (dataset de respaldo del curso)',
    'enlace':             'generado en el notebook',
    'licencia':           'uso académico',
    'filas':              1512,
    'columnas':           12,
    'una_fila_es':        'un estudiante',
    'pregunta':           '¿Qué estudiantes tienen alto riesgo de desertar, y qué '
                          'variables lo anticipan?',
    'columna_objetivo':   'deserta',
    'tipo_aprendizaje':   'clasificacion',
    'por_que_ese_tipo':   'El objetivo es una categoría binaria ya etiquetada en los '
                          'datos históricos.',
    'hay_desbalance':     'Sí: ~24 % deserta. No usaré exactitud como métrica.',
    'plan_faltantes':     'promedio_anterior y asistencia_pct -> mediana. '
                          'horas_trabajo (31 %) -> categoría "No informa".',
    'plan_categoricas':   'ciudad y programa -> one-hot. estrato -> ordinal (tiene '
                          'orden real). tiene_beca -> 0/1.',
    'columnas_a_eliminar': 'id_estudiante (identificador) y motivo_retiro (fuga de datos)',
    'si_falla':           'Un falso negativo deja sin acompañamiento a quien lo '
                          'necesitaba. Un falso positivo puede estigmatizar, así que '
                          'la alerta va a bienestar universitario, no al expediente.',
}

for k, v in ficha_ejemplo.items():
    print(f'{k:22s}: {v}')

tema                  : Deserción estudiantil en pregrado
dataset               : Registros académicos (dataset de respaldo del curso)
enlace                : generado en el notebook
licencia              : uso académico
filas                 : 1512
columnas              : 12
una_fila_es           : un estudiante
pregunta              : ¿Qué estudiantes tienen alto riesgo de desertar, y qué variables lo anticipan?
columna_objetivo      : deserta
tipo_aprendizaje      : clasificacion
por_que_ese_tipo      : El objetivo es una categoría binaria ya etiquetada en los datos históricos.
hay_desbalance        : Sí: ~24 % deserta. No usaré exactitud como métrica.
plan_faltantes        : promedio_anterior y asistencia_pct -> mediana. horas_trabajo (31 %) -> categoría "No informa".
plan_categoricas      : ciudad y programa -> one-hot. estrato -> ordinal (tiene orden real). tiene_beca -> 0/1.
columnas_a_eliminar   : id_estudiante (identificador) y motivo_retiro (fuga de datos)
si_falla           

### Ahora tú

Copia la estructura y llénala con **tu** proyecto. Sé concreto: los nombres de
columna que escribas deben existir de verdad en tu dataset.

In [36]:
# TODO — TU ficha. Esta es la entrega de hoy.

mi_ficha = {
    'nombre':              'Villegas taller 2',
    'tema':                'Riesgo reproductivo del ganado bovino en la Ganadería Olinda',
    'dataset':             'Registros consolidados de la Ganadería Olinda, una fila por vaca '
                           '(OLINDA_CLEAN.csv)',
    'enlace':              'archivo propio de la finca, sin enlace público',
    'licencia':            'uso académico autorizado por el propietario de los datos',
    'filas':               2500,
    'columnas':            15,
    'una_fila_es':         'una vaca de la Ganadería Olinda',
    'pregunta':            '¿Qué vacas tienen mayor riesgo de ser candidatas a descarte por bajo '
                           'desempeño reproductivo?',
    'columna_objetivo':    'candidata_descarte',
    'tipo_aprendizaje':    'clasificacion',
    'por_que_ese_tipo':    'El objetivo es una etiqueta binaria (candidata a descarte / no) ya '
                           'calculada en los registros históricos, no un número continuo.',
    'hay_desbalance':      'Sí: 32.5% candidatas a descarte (813 de 2500). No usaré exactitud como métrica.',
    'plan_faltantes':      'No aplica: el dataset consolidado no tiene valores nulos en ninguna columna.',
    'plan_categoricas':    'color y estado_salud -> one-hot (no tienen orden real). cachona y '
                           'perdida_cria ya están en 0/1. condicion_corporal se deja numérica '
                           '(es una escala continua, no categorías).',
    'columnas_a_eliminar': 'vaca_id (identificador), intervalo_partos_meses y '
                           'clasificacion_reproductiva (fuga de datos: son la base con la que se '
                           'calculó candidata_descarte)',
    'si_falla':            'Un falso negativo (no marcar como candidata a descarte a una vaca que '
                           'sí lo es) hace que se le siga invirtiendo alimentación y cuidado sin '
                           'que aporte crías. Un falso positivo lleva a revisar de más a una vaca '
                           'sana. Por eso el recall de la clase candidata_descarte importa más que '
                           'la precisión.',
}

for k, v in mi_ficha.items():
    print(f'{k:22s}: {v}')

nombre                : Villegas taller 2
tema                  : Riesgo reproductivo del ganado bovino en la Ganadería Olinda
dataset               : Registros consolidados de la Ganadería Olinda, una fila por vaca (OLINDA_CLEAN.csv)
enlace                : archivo propio de la finca, sin enlace público
licencia              : uso académico autorizado por el propietario de los datos
filas                 : 2500
columnas              : 15
una_fila_es           : una vaca de la Ganadería Olinda
pregunta              : ¿Qué vacas tienen mayor riesgo de ser candidatas a descarte por bajo desempeño reproductivo?
columna_objetivo      : candidata_descarte
tipo_aprendizaje      : clasificacion
por_que_ese_tipo      : El objetivo es una etiqueta binaria (candidata a descarte / no) ya calculada en los registros históricos, no un número continuo.
hay_desbalance        : Sí: 32.5% candidatas a descarte (813 de 2500). No usaré exactitud como métrica.
plan_faltantes        : No aplica: el dataset 

In [37]:
# Comprobación antes de entregar: ¿mencionas columnas que existen de verdad?
print('Columnas reales de mi dataset:')
print(list(datos.columns))

Columnas reales de mi dataset:
['vaca_id', 'edad_meses', 'color', 'cachona', 'peso_kg', 'condicion_corporal', 'estado_salud', 'edad_primer_parto_meses', 'num_partos', 'intervalo_partos_meses', 'meses_desde_ultimo_parto', 'perdida_cria', 'produccion_leche_lt_dia', 'candidata_descarte', 'clasificacion_reproductiva']


---

## Antes de irte

1. Guarda el notebook como `Taller 2 - TuApellido.ipynb`.
2. **Muéstrame en pantalla tu propuesta (Parte A) y tu ficha técnica (B4).**
   Juntas son la entrega de hoy.

## Para la Sesión 3 — viernes 28 de agosto, 5:00 PM – 9:00 PM

**Tema:** Modelos de clasificación, validación y ajuste.
Por fin entrenamos modelos, y con tus datos.

1. Aplica a tu dataset el plan de preprocesamiento que escribiste hoy.
2. Deja un notebook con los datos ya limpios y guardados
   (`datos_limpios.csv` o `.to_pickle()`).
3. Si tu proyecto es de **regresión o agrupación**, piensa además una pregunta
   de **clasificación** sobre los mismos datos: la Sesión 3 la vas a necesitar.
4. Repasa qué eran precisión, recall y matriz de confusión.

## Si quieres leer más

- Géron, A. (2022). *Hands-On Machine Learning* (3.ª ed.) — **Capítulo 2**
  (el proyecto de punta a punta) y el apartado de preparación de datos.
- Documentación de pandas: `isna`, `fillna`, `get_dummies`, `drop_duplicates`.
- scikit-learn: `SimpleImputer`, `OneHotEncoder`, `StandardScaler`.
